<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

```
Given: gs (stomatal conductance), gbc (boundary layer conductance for CO₂)
       Vcmax, J, Kc, Ko, Γ*, Rd (all temperature-adjusted)
       ca (atmospheric CO₂), oi (atmospheric O₂)
                    │
                    ▼
    ┌───────────────────────────────────┐
    │ Calculate gleaf = total CO₂       │  ← Eq. 11.75
    │ conductance (series combination)  │
    └───────────────────────────────────┘
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
   Rubisco-limited       Light-limited
   (solve quadratic      (solve quadratic
    for Ac)               for Aj)
   Eq. 11.28 + 11.78    Eq. 11.29 + 11.78
          │                   │
          └─────────┬─────────┘
                    ▼
    ┌───────────────────────────────────┐
    │ Co-limitation: blend Ac and Aj    │  ← Eq. 11.31
    │ Θ·A² - (Ac+Aj)·A + Ac·Aj = 0      │
    └───────────────────────────────────┘
                    │
                    ▼
        An = Ag - Rd                       ← Eq. 11.30
        cs = ca - An/gbc                   ← Eq. 11.75
        ci = ca - An/gleaf                 ← Eq. 11.76
```

In [0]:
#| echo: false
#| output: asis
show_doc(leaf_ci_optimization)

---

[source](https://github.com/ecamo19/plant_hydraulics/blob/main/plant_hydraulics/leaf_ci_optimization.py#L11){target="_blank" style="float:right; font-size:smaller"}

### leaf_ci_optimization

```python

def leaf_ci_optimization(
    atmos:Atmos, # Atmospheric forcing variables:
- o2air : float
    Atmospheric O2 concentration (mmol/mol).
- co2air : float
    Atmospheric CO2 concentration (umol/mol).
    leaf:Leaf, # Leaf physiological parameters:
- c3psn : int
    Photosynthetic pathway: 1 = C3, 0 = C4 plant.
- colim : int
    Photosynthesis co-limitation: 0 = no, 1 = yes.
- colim_c3 : float
    Empirical curvature parameter for C3 co-limitation (-).
- colim_c4a : float
    Empirical curvature parameter for C4 co-limitation (-).
- colim_c4b : float
    Empirical curvature parameter for C4 co-limitation (-).
- qe_c4 : float
    C4 quantum yield (mol CO2 / mol photons).
    flux:Flux, # Flux variables with the following inputs:
- vcmax : float
    Maximum carboxylation rate (umol/m2/s).
- cp : float
    CO2 compensation point (umol/mol).
- kc : float
    Michaelis-Menten constant for CO2 (umol/mol).
- ko : float
    Michaelis-Menten constant for O2 (mmol/mol).
- je : float
    Electron transport rate (umol/m2/s).
- kp_c4 : float
    C4 initial slope of CO2 response curve (mol/m2/s).
- gs : float
    Leaf stomatal conductance (mol H2O/m2 leaf/s).
- gbc : float
    Leaf boundary layer conductance for CO2 (mol CO2/m2 leaf/s).
- apar : float
    Leaf absorbed PAR (umol photon/m2 leaf/s).
- rd : float
    Leaf respiration rate (umol CO2/m2 leaf/s).
)->Flux: # Updated flux object with the following attributes:
- ac : float
    Rubisco-limited gross photosynthesis (umol CO2/m2 leaf/s).
- aj : float
    RuBP regeneration-limited gross photosynthesis (umol CO2/m2 leaf/s).
- ap : float
    Product-limited (C3) or CO2-limited (C4) gross photosynthesis
    (umol CO2/m2 leaf/s).
- ag : float
    Leaf gross photosynthesis (umol CO2/m2 leaf/s).
- an : float
    Leaf net photosynthesis (umol CO2/m2 leaf/s).
- cs : float
    Leaf surface CO2 concentration (umol/mol).
- ci : float
    Leaf intercellular CO2 concentration (umol/mol).


```

*Calculate leaf photosynthesis for a specified stomatal conductance,*
then calculate Ci from the diffusion equation.

__Phys 101:__ 

- Carboxilation rate: How many reactions (CO2 + RuBP -> 3-PGA) happen per 
unit of leaf area per second.

- Vcmax: Maximum Carboxilation rate

Meaning of Θ (aka the co-limitation curvature factor): Θ asks, when the 
plant transitions from being Rubisco-limited to light-limited, is that 
transition sharp (Θ near 1) or gradual (Θ near 0)? 

A sharp transition means one process dominates at any given moment. 
A gradual transition means both processes are always partially 
limiting simultaneously. The default value in the code is 0.98 

__The function__

This routine uses a quadratic equation to solve for net photosynthesis
(An). A general equation for C3 photosynthesis is:

         a*(Ci - Cp)
    An = ----------- - Rd
          e*Ci + d

where:

    An = Net leaf photosynthesis (umol CO2/m2/s)
    Rd = Leaf respiration (umol CO2/m2/s)
    Ci = Intercellular CO2 concentration (umol/mol)
    Cp = CO2 compensation point (umol/mol)

- Rubisco-limited photosynthesis (Ac):

    a = Vcmax,  e = 1,  d = Kc * (1 + Oi / Ko)

- RuBP regeneration-limited photosynthesis (Aj):

    a = J,  e = 4,  d = 8 * Cp

where:

    Vcmax = Maximum carboxylation rate (umol/m2/s)
    Kc    = Michaelis-Menten constant for CO2 (umol/mol)
    Ko    = Michaelis-Menten constant for O2 (mmol/mol)
    Oi    = Intercellular O2 concentration (mmol/mol)
    J     = Electron transport rate (umol/m2/s)

Ci is calculated from the diffusion equation:

                      1.4   1.6
    An = (Ca - Ci) / (--- + ---)
                      gb    gs

               1.4   1.6
    Ci = Ca - (--- + ---)*An
                gb    gs

where:

    Ca  = Atmospheric CO2 concentration (umol/mol)
    gb  = Leaf boundary layer conductance (mol H2O/m2/s)
    gs  = Leaf stomatal conductance (mol H2O/m2/s)
    1.4 = Corrects gb for the diffusivity of CO2 compared with H2O
    1.6 = Corrects gs for the diffusivity of CO2 compared with H2O

The resulting quadratic equation is: a*An^2 + b*An + c = 0, which
is solved for An. Correct solution is the smaller of the two roots.

A similar approach is used for C4 photosynthesis.

The total leaf conductance for CO2 (gleaf, mol CO2/m2/s) is computed
from the boundary layer conductance gbc (mol CO2/m2/s) and stomatal
conductance gs (mol H2O/m2/s) acting in series:

    gleaf = 1 / (1/gbc + 1.6/gs)

where the factor 1.6 converts gs from H2O to CO2 basis.

__Parameters:__

- Atmos: Atmospheric forcing variables

    - o2air: Atmospheric O2 concentration (mmol/mol).
    - co2air: Atmospheric CO2 concentration (umol/mol).

- Leaf: Leaf physiological parameters

    - c3psn: Photosynthetic pathway: 1 = C3, 0 = C4 plant.
    - colim: Photosynthesis co-limitation: 0 = no, 1 = yes.
    - colim_c3: Empirical curvature parameter for C3 co-limitation (-).
    - colim_c4a: Empirical curvature parameter for C4 co-limitation (-).
    - colim_c4b: Empirical curvature parameter for C4 co-limitation (-).
    - qe_c4: C4 quantum yield (mol CO2 / mol photons).

- Flux: Flux variables with the following inputs

    - vcmax: Maximum carboxylation rate (umol/m2/s).
    - cp: CO2 compensation point (umol/mol).
    - kc: Michaelis-Menten constant for CO2 (umol/mol).
    - ko: Michaelis-Menten constant for O2 (mmol/mol).
    - je: Electron transport rate (umol/m2/s).
    - kp_c4: C4 initial slope of CO2 response curve (mol/m2/s).
    - gs: Leaf stomatal conductance (mol H2O/m2 leaf/s).
    - gbc: Leaf boundary layer conductance for CO2 (mol CO2/m2 leaf/s).
    - apar: Leaf absorbed PAR (umol photon/m2 leaf/s).
    - rd: Leaf respiration rate (umol CO2/m2 leaf/s).

__Returns:__

- Flux: Updated flux object with the following attributes

    - ac: Rubisco-limited gross photosynthesis (umol CO2/m2 leaf/s).
    - aj: RuBP regeneration-limited gross photosynthesis (umol CO2/m2 leaf/s).
    - ap: Product-limited (C3) or CO2-limited (C4) gross photosynthesis (umol CO2/m2 leaf/s).
    - ag: Leaf gross photosynthesis (umol CO2/m2 leaf/s).
    - an: Leaf net photosynthesis (umol CO2/m2 leaf/s).
    - cs: Leaf surface CO2 concentration (umol/mol).
    - ci: Leaf intercellular CO2 concentration (umol/mol).